# Diffusion Manifold Projection: Sample Efficiency Experiment

**Hypothesis**: If you train a diffusion model (teacher) on a large dataset, then "reconstruct" samples by partial noising + denoising (projecting them onto the learned manifold), a student model trained on those reconstructed samples achieves better FID than a student trained on the same number of raw samples.

This is the diffusion analogue of a known result for autoencoders.

## Setup
- **Dataset**: Fashion-MNIST (60k training, 10k test)
- **Architecture**: Lightweight U-Net DDPM
- **GPU**: This notebook is designed for Colab's free T4 GPU

**Runtime**: ~3-5 hours total on a T4 (teacher ~45 min, students ~2-3 hours, sampling/FID ~1 hour)

In [ ]:
# Install dependencies
!pip install -q clean-fid

In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
CFG = dict(
    # Paths
    output_dir="./results",
    data_dir="./data",

    # Noise schedule
    num_timesteps=1000,
    beta_start=1e-4,
    beta_end=0.02,
    schedule="linear",

    # U-Net architecture
    base_channels=64,
    channel_mults=(1, 2, 2),
    num_res_blocks=2,
    attention_resolutions=(2,),
    dropout=0.1,
    time_embed_dim=256,

    # Training
    lr=2e-4,
    batch_size=128,
    ema_decay=0.9999,
    grad_clip=1.0,

    # Teacher
    teacher_epochs=80,
    teacher_patience=10,

    # Students
    student_steps=15000,

    # Projection / generation
    t_proj_values=(200,),
    n_values=(500, 1000, 2000, 5000, 10000, 20000),

    # Evaluation
    eval_n_samples=10000,
    sampling_batch_size=256,

    # Misc
    seed=42,
    log_every=200,
)

# --- Uncomment below for a quick smoke test (~10 min) ---
# CFG.update(dict(
#     teacher_epochs=3,
#     teacher_patience=100,
#     student_steps=300,
#     n_values=(500, 1000),
#     eval_n_samples=1000,
#     base_channels=32,
#     num_res_blocks=1,
#     attention_resolutions=(),
# ))

os.makedirs(CFG["output_dir"], exist_ok=True)
print("Config ready.")

In [ ]:
# Reproducibility
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

## Noise Schedule

In [ ]:
class NoiseSchedule:
    """Precomputes diffusion coefficients for a linear or cosine schedule."""

    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02,
                 schedule="linear", device="cpu"):
        self.T = num_timesteps
        if schedule == "linear":
            betas = torch.linspace(beta_start, beta_end, num_timesteps, dtype=torch.float64)
        elif schedule == "cosine":
            steps = torch.arange(num_timesteps + 1, dtype=torch.float64)
            alpha_bar = torch.cos(((steps / num_timesteps) + 0.008) / 1.008 * (math.pi / 2)) ** 2
            alpha_bar = alpha_bar / alpha_bar[0]
            betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
            betas = betas.clamp(max=0.999)
        else:
            raise ValueError(f"Unknown schedule: {schedule}")

        alphas = 1.0 - betas
        alpha_cumprod = torch.cumprod(alphas, dim=0)
        alpha_cumprod_prev = torch.cat([torch.ones(1, dtype=torch.float64), alpha_cumprod[:-1]])

        self.betas = betas.float().to(device)
        self.alphas = alphas.float().to(device)
        self.alpha_cumprod = alpha_cumprod.float().to(device)
        self.alpha_cumprod_prev = alpha_cumprod_prev.float().to(device)
        self.sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod).float().to(device)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod).float().to(device)

        self.posterior_variance = (
            betas * (1.0 - alpha_cumprod_prev) / (1.0 - alpha_cumprod)
        ).float().to(device)
        self.posterior_log_variance_clipped = torch.log(
            torch.clamp(self.posterior_variance, min=1e-20)
        ).float().to(device)
        self.posterior_mean_coef1 = (
            betas * torch.sqrt(alpha_cumprod_prev) / (1.0 - alpha_cumprod)
        ).float().to(device)
        self.posterior_mean_coef2 = (
            (1.0 - alpha_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alpha_cumprod)
        ).float().to(device)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_alpha * x0 + sqrt_one_minus * noise, noise

    def predict_x0_from_eps(self, x_t, t, eps):
        sqrt_alpha = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        return (x_t - sqrt_one_minus * eps) / sqrt_alpha

    def q_posterior_mean_variance(self, x0, x_t, t):
        mean = (self.posterior_mean_coef1[t].view(-1, 1, 1, 1) * x0 +
                self.posterior_mean_coef2[t].view(-1, 1, 1, 1) * x_t)
        var = self.posterior_variance[t].view(-1, 1, 1, 1)
        log_var = self.posterior_log_variance_clipped[t].view(-1, 1, 1, 1)
        return mean, var, log_var

ns = NoiseSchedule(
    num_timesteps=CFG["num_timesteps"],
    beta_start=CFG["beta_start"],
    beta_end=CFG["beta_end"],
    schedule=CFG["schedule"],
    device=device,
)
print(f"Noise schedule ready (T={ns.T})")

## U-Net Architecture

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(32, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.norm2 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class AttentionBlock(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(min(32, ch), ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        self.num_heads = num_heads

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, C // self.num_heads, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        scale = (C // self.num_heads) ** -0.5
        attn = torch.einsum("bhci,bhcj->bhij", q, k) * scale
        attn = attn.softmax(dim=-1)
        out = torch.einsum("bhij,bhcj->bhci", attn, v)
        out = out.reshape(B, C, H, W)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, ch_mults=(1, 2, 2),
                 num_res_blocks=2, attn_resolutions=(2,), dropout=0.1,
                 time_embed_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(base_ch),
            nn.Linear(base_ch, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim),
        )
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        channels = [base_ch]
        ch = base_ch

        # Encoder
        self.down_blocks = nn.ModuleList()
        for level, mult in enumerate(ch_mults):
            out_ch = base_ch * mult
            for _ in range(num_res_blocks):
                layers = [ResBlock(ch, out_ch, time_embed_dim, dropout)]
                if level in attn_resolutions:
                    layers.append(AttentionBlock(out_ch))
                self.down_blocks.append(nn.ModuleList(layers))
                channels.append(out_ch)
                ch = out_ch
            if level < len(ch_mults) - 1:
                self.down_blocks.append(nn.ModuleList([Downsample(ch)]))
                channels.append(ch)

        # Middle
        self.mid_block1 = ResBlock(ch, ch, time_embed_dim, dropout)
        self.mid_attn = AttentionBlock(ch)
        self.mid_block2 = ResBlock(ch, ch, time_embed_dim, dropout)

        # Decoder
        self.up_blocks = nn.ModuleList()
        for level, mult in reversed(list(enumerate(ch_mults))):
            out_ch = base_ch * mult
            for i in range(num_res_blocks + 1):
                skip_ch = channels.pop()
                layers = [ResBlock(ch + skip_ch, out_ch, time_embed_dim, dropout)]
                if level in attn_resolutions:
                    layers.append(AttentionBlock(out_ch))
                self.up_blocks.append(nn.ModuleList(layers))
                ch = out_ch
            if level > 0:
                self.up_blocks.append(nn.ModuleList([Upsample(ch)]))

        self.final_norm = nn.GroupNorm(min(32, ch), ch)
        self.final_conv = nn.Conv2d(ch, in_ch, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.init_conv(x)
        skips = [h]

        for block in self.down_blocks:
            if isinstance(block[0], Downsample):
                h = block[0](h)
                skips.append(h)
            else:
                for layer in block:
                    h = layer(h, t_emb) if isinstance(layer, ResBlock) else layer(h)
                skips.append(h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        for block in self.up_blocks:
            if isinstance(block[0], Upsample):
                h = block[0](h)
            else:
                s = skips.pop()
                h = torch.cat([h, s], dim=1)
                for layer in block:
                    h = layer(h, t_emb) if isinstance(layer, ResBlock) else layer(h)

        return self.final_conv(F.silu(self.final_norm(h)))


def build_model():
    model = UNet(
        in_ch=1,
        base_ch=CFG["base_channels"],
        ch_mults=CFG["channel_mults"],
        num_res_blocks=CFG["num_res_blocks"],
        attn_resolutions=CFG["attention_resolutions"],
        dropout=CFG["dropout"],
        time_embed_dim=CFG["time_embed_dim"],
    ).to(device)
    return model

# Quick check
m = build_model()
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
del m

## EMA & Training Utilities

In [ ]:
class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)

    def apply(self, model):
        backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
        return backup

    def restore(self, model, backup):
        model.load_state_dict(backup)


def make_loader(images, batch_size, shuffle=True):
    ds = TensorDataset(images)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=True, num_workers=0)


def train_ddpm(model, dataloader, num_epochs=None, num_steps=None,
               desc="Training", val_loader=None, patience=None):
    """Train a DDPM. Epoch-based (teacher) or step-based (students)."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
    ema = EMA(model, decay=CFG["ema_decay"])
    T = ns.T

    history = {"train_loss": [], "val_loss": [], "steps": []}
    global_step = 0
    best_val_loss = float("inf")
    patience_counter = 0

    def step(batch):
        nonlocal global_step
        x0 = batch[0].to(device) if isinstance(batch, (list, tuple)) else batch.to(device)
        B = x0.shape[0]
        t = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(x0)
        x_t, _ = ns.q_sample(x0, t, noise)
        pred = model(x_t, t)
        loss = F.mse_loss(pred, noise)
        optimizer.zero_grad()
        loss.backward()
        if CFG["grad_clip"] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()
        ema.update(model)
        global_step += 1
        return loss.item()

    if num_epochs is not None:
        for epoch in range(num_epochs):
            model.train()
            losses = []
            pbar = tqdm(dataloader, desc=f"{desc} ep {epoch+1}/{num_epochs}", leave=False)
            for batch in pbar:
                l = step(batch)
                losses.append(l)
                if global_step % CFG["log_every"] == 0:
                    history["train_loss"].append(l)
                    history["steps"].append(global_step)
                pbar.set_postfix(loss=f"{l:.4f}")
            avg_train = np.mean(losses)

            if val_loader is not None:
                model.eval()
                vl = []
                with torch.no_grad():
                    for batch in val_loader:
                        x0 = batch[0].to(device)
                        B = x0.shape[0]
                        t = torch.randint(0, T, (B,), device=device)
                        noise = torch.randn_like(x0)
                        x_t, _ = ns.q_sample(x0, t, noise)
                        vl.append(F.mse_loss(model(x_t, t), noise).item())
                avg_val = np.mean(vl)
                history["val_loss"].append(avg_val)
                print(f"  Epoch {epoch+1}: train={avg_train:.4f} val={avg_val:.4f}")
                if patience is not None:
                    if avg_val < best_val_loss - 1e-5:
                        best_val_loss = avg_val
                        patience_counter = 0
                    else:
                        patience_counter += 1
                        if patience_counter >= patience:
                            print(f"  Early stopping at epoch {epoch+1}")
                            break
            else:
                print(f"  Epoch {epoch+1}: train={avg_train:.4f}")

    elif num_steps is not None:
        model.train()
        data_iter = iter(dataloader)
        pbar = tqdm(range(num_steps), desc=desc, leave=False)
        for s in pbar:
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)
            l = step(batch)
            if s % CFG["log_every"] == 0:
                history["train_loss"].append(l)
                history["steps"].append(global_step)
            if s % 500 == 0:
                pbar.set_postfix(loss=f"{l:.4f}")

    return history, ema

## Sampling Functions

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples, batch_size=256):
    """Full reverse process: pure noise → generated images."""
    all_samples = []
    for start in range(0, n_samples, batch_size):
        B = min(batch_size, n_samples - start)
        x = torch.randn(B, 1, 28, 28, device=device)
        for t in tqdm(reversed(range(ns.T)), total=ns.T,
                      desc=f"Sampling {start}-{start+B}", leave=False):
            tb = torch.full((B,), t, device=device, dtype=torch.long)
            eps = model(x, tb)
            x0_pred = ns.predict_x0_from_eps(x, tb, eps).clamp(-1, 1)
            mean, var, log_var = ns.q_posterior_mean_variance(x0_pred, x, tb)
            noise = torch.randn_like(x) if t > 0 else 0.0
            x = mean + torch.exp(0.5 * log_var) * noise
        all_samples.append(x.cpu())
    return torch.cat(all_samples)


@torch.no_grad()
def ddpm_project(model, images, t_proj, batch_size=256):
    """Partial noise → denoise: the diffusion analogue of autoencoder reconstruction."""
    all_out = []
    for start in range(0, len(images), batch_size):
        x0 = images[start:start + batch_size].to(device)
        B = x0.shape[0]
        tb = torch.full((B,), t_proj, device=device, dtype=torch.long)
        x_t, _ = ns.q_sample(x0, tb)

        x = x_t
        for t in tqdm(reversed(range(t_proj)), total=t_proj,
                      desc=f"Projecting {start}-{start+B}", leave=False):
            tb2 = torch.full((B,), t, device=device, dtype=torch.long)
            eps = model(x, tb2)
            x0_pred = ns.predict_x0_from_eps(x, tb2, eps).clamp(-1, 1)
            mean, var, log_var = ns.q_posterior_mean_variance(x0_pred, x, tb2)
            noise = torch.randn_like(x) if t > 0 else 0.0
            x = mean + torch.exp(0.5 * log_var) * noise
        all_out.append(x.cpu())
    return torch.cat(all_out)


def save_images_for_fid(images, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    images = (images.clamp(-1, 1) + 1) / 2
    for i, img in enumerate(images):
        save_image(img, os.path.join(out_dir, f"{i:05d}.png"))

## Load Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_ds = datasets.FashionMNIST(CFG["data_dir"], train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(CFG["data_dir"], train=False, download=True, transform=transform)

# Extract all as tensors
def to_tensors(ds):
    loader = DataLoader(ds, batch_size=1000, shuffle=False, num_workers=0)
    imgs, labs = [], []
    for x, y in loader:
        imgs.append(x); labs.append(y)
    return torch.cat(imgs), torch.cat(labs)

all_train_imgs, all_train_labels = to_tensors(train_ds)
test_imgs, test_labels = to_tensors(test_ds)

# Split: 54k train, 6k val
perm = torch.randperm(len(all_train_imgs))
val_imgs = all_train_imgs[perm[:6000]]
teacher_train_imgs = all_train_imgs[perm[6000:]]

print(f"Teacher train: {teacher_train_imgs.shape}")
print(f"Validation: {val_imgs.shape}")
print(f"Test: {test_imgs.shape}")

# Save FID reference images
ref_dir = os.path.join(CFG["output_dir"], "fid_reference")
if not os.path.exists(os.path.join(ref_dir, "00000.png")):
    print("Saving reference images for FID...")
    save_images_for_fid(test_imgs, ref_dir)
    print("Done.")

## Step 1: Train Teacher

In [ ]:
teacher_path = os.path.join(CFG["output_dir"], "teacher.pt")

teacher = build_model()
print(f"Parameters: {sum(p.numel() for p in teacher.parameters() if p.requires_grad):,}")

if os.path.exists(teacher_path):
    print("Loading saved teacher...")
    ckpt = torch.load(teacher_path, map_location=device, weights_only=True)
    teacher.load_state_dict(ckpt["ema"])
    print("Loaded.")
else:
    train_loader = make_loader(teacher_train_imgs, CFG["batch_size"])
    val_loader = make_loader(val_imgs, CFG["batch_size"], shuffle=False)

    t0 = time.time()
    history, ema = train_ddpm(
        teacher, train_loader,
        num_epochs=CFG["teacher_epochs"],
        desc="Teacher",
        val_loader=val_loader,
        patience=CFG["teacher_patience"],
    )
    elapsed = time.time() - t0
    print(f"\nTeacher training: {elapsed/60:.1f} min")

    ema.apply(teacher)
    torch.save({"model": teacher.state_dict(), "ema": teacher.state_dict(),
                "history": history}, teacher_path)
    print(f"Saved to {teacher_path}")

    # Plot
    if history["train_loss"]:
        plt.figure(figsize=(8, 4))
        plt.plot(history["steps"], history["train_loss"], alpha=0.5, label="train")
        if history["val_loss"]:
            vx = np.linspace(0, history["steps"][-1], len(history["val_loss"]))
            plt.plot(vx, history["val_loss"], label="val")
        plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("Teacher Loss")
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(CFG["output_dir"], "teacher_loss.png"), dpi=150)
        plt.show()

In [ ]:
# Quick visual check: generate a few teacher samples
teacher.eval()
demo = ddpm_sample(teacher, 16, batch_size=16)
grid = make_grid((demo + 1) / 2, nrow=4)
plt.figure(figsize=(4, 4))
plt.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
plt.title("Teacher samples"); plt.axis("off"); plt.show()

## Teacher FID

In [ ]:
from cleanfid import fid

teacher_samples_dir = os.path.join(CFG["output_dir"], "samples_teacher")
if not os.path.exists(os.path.join(teacher_samples_dir, "00000.png")):
    print(f"Generating {CFG['eval_n_samples']} teacher samples for FID...")
    teacher_samples = ddpm_sample(teacher, CFG["eval_n_samples"],
                                  batch_size=CFG["sampling_batch_size"])
    save_images_for_fid(teacher_samples, teacher_samples_dir)
    del teacher_samples

device_str = "cuda" if torch.cuda.is_available() else "cpu"
teacher_fid = fid.compute_fid(teacher_samples_dir, ref_dir,
                               mode="clean", device=torch.device(device_str))
print(f"\nTeacher FID: {teacher_fid:.2f}")

## Steps 2 & 3: Project/Generate Data, Train Students

In [ ]:
results = {"teacher_fid": teacher_fid, "experiments": []}

for t_proj in CFG["t_proj_values"]:
    print(f"\n{'='*60}")
    print(f"t_proj = {t_proj}")
    print(f"{'='*60}")

    for n in CFG["n_values"]:
        print(f"\n--- n = {n} ---")
        row = {"n": n, "t_proj": t_proj}

        # Subsample raw data
        raw_idx = torch.randperm(len(all_train_imgs))[:n]
        raw_imgs = all_train_imgs[raw_idx]

        # Projected data
        print(f"  Projecting {n} images at t_proj={t_proj}...")
        proj_imgs = ddpm_project(teacher, raw_imgs, t_proj,
                                 batch_size=CFG["sampling_batch_size"])

        # Generated data (from pure noise)
        print(f"  Generating {n} images from scratch...")
        gen_imgs = ddpm_sample(teacher, n,
                               batch_size=CFG["sampling_batch_size"])

        # Train three students
        for condition, data in [("raw", raw_imgs), ("proj", proj_imgs), ("gen", gen_imgs)]:
            print(f"  Training student_{condition}_{n}...")
            student = build_model()
            loader = make_loader(data, min(CFG["batch_size"], n))
            _, ema = train_ddpm(
                student, loader,
                num_steps=CFG["student_steps"],
                desc=f"student_{condition}_{n}",
            )
            ema.apply(student)
            student.eval()

            # FID
            sample_dir = os.path.join(CFG["output_dir"],
                                      f"samples_{condition}_{n}_t{t_proj}")
            print(f"  Generating {CFG['eval_n_samples']} samples for FID...")
            samples = ddpm_sample(student, CFG["eval_n_samples"],
                                  batch_size=CFG["sampling_batch_size"])
            save_images_for_fid(samples, sample_dir)
            fid_score = fid.compute_fid(sample_dir, ref_dir,
                                         mode="clean", device=torch.device(device_str))
            row[f"fid_{condition}"] = fid_score
            print(f"  FID {condition}: {fid_score:.2f}")

            del student, samples
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        results["experiments"].append(row)

        # Intermediate save
        with open(os.path.join(CFG["output_dir"], "results.json"), "w") as f:
            json.dump(results, f, indent=2)

print("\nAll students trained!")

## Step 4: Results

In [ ]:
# Print table
header = f"{'n':>7} | {'FID raw':>10} | {'FID proj':>10} | {'FID gen':>10} | {'Teacher':>10}"
print(header)
print("-" * len(header))
for row in results["experiments"]:
    print(f"{row['n']:>7} | {row['fid_raw']:>10.2f} | {row['fid_proj']:>10.2f} | "
          f"{row['fid_gen']:>10.2f} | {teacher_fid:>10.2f}")

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ns_vals = [r["n"] for r in results["experiments"]]
fid_raw = [r["fid_raw"] for r in results["experiments"]]
fid_proj = [r["fid_proj"] for r in results["experiments"]]
fid_gen = [r["fid_gen"] for r in results["experiments"]]

ax.plot(ns_vals, fid_raw, "o-", label="Student (raw data)",
        color="#d62728", linewidth=2, markersize=8)
ax.plot(ns_vals, fid_proj, "s-",
        label=f"Student (projected, t={CFG['t_proj_values'][0]})",
        color="#2ca02c", linewidth=2, markersize=8)
ax.plot(ns_vals, fid_gen, "^-", label="Student (generated)",
        color="#1f77b4", linewidth=2, markersize=8)
ax.axhline(y=teacher_fid, linestyle="--", color="gray", alpha=0.7,
           label=f"Teacher (FID={teacher_fid:.1f})")

ax.set_xlabel("Number of training samples (n)", fontsize=12)
ax.set_ylabel("FID (lower is better)", fontsize=12)
ax.set_title("Diffusion Manifold Projection: Sample Efficiency", fontsize=14)
ax.legend(fontsize=11)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.set_xticks(ns_vals)
ax.set_xticklabels([str(n) for n in ns_vals])
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "fid_comparison.png"), dpi=200)
plt.show()

# Save final results
with open(os.path.join(CFG["output_dir"], "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Experiment complete!")